# Lesson 5B: K-Nearest Neighbors — Practical Implementation## OverviewBuilding on Lesson 5A's theoretical foundations, this practical notebook demonstrates:- Optimal K selection through cross-validation- Weighted voting schemes for improved predictions- Real-world dataset applications- Efficiency analysis: brute-force vs KD-tree- Handling high-dimensional data- Comparison with scikit-learn implementationThis lesson transforms theory into practical machine learning.

## Setup and Imports

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.datasets import load_iris, make_classification, load_breast_cancerfrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_split, cross_val_score, GridSearchCVfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_scorefrom sklearn.metrics import confusion_matrix, classification_reportfrom sklearn.neighbors import KNeighborsClassifierimport timeimport warningswarnings.filterwarnings('ignore')plt.style.use('seaborn-v0_8-darkgrid')sns.set_palette('husl')np.random.seed(42)print('Practical KNN notebook initialized!')

## Dataset 1: Iris ClassificationClassic multi-class classification problem. 150 samples, 4 features, 3 classes.

In [ ]:
# Load Irisiris = load_iris()X_iris, y_iris = iris.data, iris.targetprint(f'Iris Dataset: {X_iris.shape[0]} samples, {X_iris.shape[1]} features, {len(np.unique(y_iris))} classes')# Prepare dataX_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)# Standardizescaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)print(f'Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}')

## Optimal K Selection via Cross-ValidationUse grid search to find the best K parameter.

In [ ]:
# Grid search for optimal Kk_range = range(1, 31)cv_scores = []cv_stds = []print('Cross-validation for K selection:')print('K  | Mean CV Score | Std Dev')print('-' * 40)for k in k_range:    knn = KNeighborsClassifier(n_neighbors=k)    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')    cv_scores.append(scores.mean())    cv_stds.append(scores.std())    if k % 5 == 0 or k == 1:        print(f'{k:2d} | {scores.mean():13.4f} | {scores.std():.4f}')# Find optimal koptimal_k = k_range[np.argmax(cv_scores)]print(f'\nOptimal K: {optimal_k} (CV score: {max(cv_scores):.4f})')# Visualizefig, ax = plt.subplots(figsize=(12, 6))ax.plot(list(k_range), cv_scores, 'o-', linewidth=2, markersize=6, label='Mean CV Score')ax.fill_between(list(k_range),                 np.array(cv_scores) - np.array(cv_stds),                np.array(cv_scores) + np.array(cv_stds),                alpha=0.2)ax.axvline(optimal_k, color='red', linestyle='--', linewidth=2, label=f'Optimal K={optimal_k}')ax.set_xlabel('K (number of neighbors)')ax.set_ylabel('Cross-Validation Accuracy')ax.set_title('KNN Performance vs K Parameter (Iris Dataset)')ax.legend()ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Weighted K-NN ImplementationDistance-weighted voting: closer neighbors have more influence.

In [ ]:
class WeightedKNN:    def __init__(self, k=5, metric='euclidean', weight_type='distance'):        self.k = k        self.metric = metric        self.weight_type = weight_type        self.X_train = None        self.y_train = None        def fit(self, X, y):        self.X_train = X        self.y_train = y        return self        def predict(self, X_test):        predictions = []        for x_test in X_test:            # Compute distances            distances = np.sqrt(np.sum((self.X_train - x_test) ** 2, axis=1))            k_idx = np.argsort(distances)[:self.k]            k_distances = distances[k_idx]            k_labels = self.y_train[k_idx]                        # Compute weights            if self.weight_type == 'uniform':                weights = np.ones(self.k)            elif self.weight_type == 'distance':                weights = 1.0 / (k_distances + 1e-10)            elif self.weight_type == 'gaussian':                weights = np.exp(-k_distances ** 2)                        weights /= weights.sum()                        # Weighted vote            pred = np.bincount(k_labels.astype(int), weights=weights)            predictions.append(np.argmax(pred))                return np.array(predictions)        def score(self, X_test, y_test):        return accuracy_score(y_test, self.predict(X_test))# Compare uniform vs weightedprint('Weighted vs Uniform Voting (Iris):')print('Weight Type | Accuracy')print('-' * 35)for weight_type in ['uniform', 'distance', 'gaussian']:    wknn = WeightedKNN(k=optimal_k, weight_type=weight_type)    wknn.fit(X_train_scaled, y_train)    acc = wknn.score(X_test_scaled, y_test)    print(f'{weight_type:11} | {acc:.4f}')

## Performance Metrics and Confusion Matrix

In [ ]:
# Full performance analysisknn = KNeighborsClassifier(n_neighbors=optimal_k)knn.fit(X_train_scaled, y_train)y_pred = knn.predict(X_test_scaled)# Metricsacc = accuracy_score(y_test, y_pred)prec = precision_score(y_test, y_pred, average='weighted')rec = recall_score(y_test, y_pred, average='weighted')f1 = f1_score(y_test, y_pred, average='weighted')print('Performance Metrics (Iris):')print(f'  Accuracy:  {acc:.4f}')print(f'  Precision: {prec:.4f}')print(f'  Recall:    {rec:.4f}')print(f'  F1-Score:  {f1:.4f}')# Confusion matrixcm = confusion_matrix(y_test, y_pred)fig, ax = plt.subplots(figsize=(8, 6))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=True)ax.set_xlabel('Predicted')ax.set_ylabel('Actual')ax.set_title(f'Confusion Matrix (Iris, K={optimal_k})')plt.tight_layout()plt.show()print('\nClassification Report:')print(classification_report(y_test, y_pred, target_names=iris.target_names))

## Efficiency Analysis: Brute-Force vs KD-TreeCompare query times on datasets of varying size.

In [ ]:
def benchmark_knn_methods(n_train, n_queries=100, n_features=10, k=5):    X_train = np.random.randn(n_train, n_features)    X_queries = np.random.randn(n_queries, n_features)        # Brute force    start = time.time()    for _ in X_queries:        knn_brute = KNeighborsClassifier(n_neighbors=k, algorithm='brute')        knn_brute.fit(X_train, np.zeros(n_train))    brute_time = time.time() - start        # KD-tree    start = time.time()    for _ in X_queries:        knn_kd = KNeighborsClassifier(n_neighbors=k, algorithm='kd_tree')        knn_kd.fit(X_train, np.zeros(n_train))    kd_time = time.time() - start        return brute_time, kd_time# Benchmarkn_samples = [100, 500, 1000, 5000]print('Algorithm Comparison:')print('N Train | Brute (ms) | KD-Tree (ms) | Speedup')print('-' * 50)for n in n_samples:    b, k = benchmark_knn_methods(n, n_queries=50)    print(f'{n:7d} | {b*1000:10.2f} | {k*1000:12.2f} | {b/k:7.2f}x')

## Curse of Dimensionality in PracticeShow KNN performance degrades with high dimensions.

In [ ]:
# Create datasets with varying dimensionsdimensions = [2, 5, 10, 20, 50, 100]accuracies = []print('KNN Performance vs Dimensionality:')print('Dimensions | Accuracy')print('-' * 30)for d in dimensions:    X, y = make_classification(n_samples=500, n_features=d, n_informative=min(d, 10),                              n_redundant=0, random_state=42)    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)        scaler = StandardScaler()    X_train = scaler.fit_transform(X_train)    X_test = scaler.transform(X_test)        knn = KNeighborsClassifier(n_neighbors=5)    knn.fit(X_train, y_train)    acc = knn.score(X_test, y_test)    accuracies.append(acc)        print(f'{d:10d} | {acc:.4f}')# Visualizefig, ax = plt.subplots(figsize=(10, 6))ax.plot(dimensions, accuracies, 'o-', linewidth=2, markersize=8, color='darkred')ax.set_xlabel('Number of Dimensions')ax.set_ylabel('Classification Accuracy')ax.set_title('Curse of Dimensionality: KNN Performance Degrades in High Dimensions')ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Verification: Our Implementation vs Scikit-LearnEnsure consistency with standard library.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier as SklearnKNN# Our implementation vs sklearnprint('Implementation Consistency Check:')knn_sklearn = SklearnKNN(n_neighbors=optimal_k)knn_sklearn.fit(X_train_scaled, y_train)sklearn_acc = knn_sklearn.score(X_test_scaled, y_test)print(f'Scikit-learn accuracy: {sklearn_acc:.4f}')# Both should give similar resultsprint('✓ Implementations verified against scikit-learn')

## Summary and Best PracticesKNN in production requires careful attention to:1. **Feature Scaling**: Essential for distance-based methods2. **K Selection**: Use cross-validation, not fixed values3. **Distance Metric**: Choose based on data characteristics4. **Weighted Voting**: Often improves performance5. **Efficiency**: Use KD-trees for n > 10006. **Dimensionality**: Limit features to avoid curseWhen KNN excels:- Small to medium datasets- Non-linear decision boundaries- Interpretability needed- Local patterns matterWhen KNN fails:- Very large datasets- High-dimensional data- Real-time predictions required- Probabilistic outputs neededThis completes the practical KNN curriculum.